In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import glob
import re
from pathlib import Path
from datetime import datetime

### Read the contour points

In [3]:
def read_contour(contour_file = "contour30sp.csv"):
    df = pd.read_csv(contour_file)

    ds = xr.Dataset(
        coords={
            "point": np.arange(len(df))
        },
        data_vars={
            "lon":      ("point", df["contour30s_lon"].to_numpy()),
            "lat":      ("point", df["contour30s_lat"].to_numpy()),
            "x":        ("point", df["contour30s_x"].to_numpy()),
            "y":        ("point", df["contour30s_y"].to_numpy()),
            "normal_x": ("point", df["normals30_x"].to_numpy()),
            "normal_y": ("point", df["normals30_y"].to_numpy()),
        }
    )
    return ds

In [4]:
contour_ds = read_contour()
contour_ds

<xarray.Dataset> Size: 24kB
Dimensions:   (point: 425)
Coordinates:
  * point     (point) int64 3kB 0 1 2 3 4 5 6 7 ... 418 419 420 421 422 423 424
Data variables:
    lon       (point) float64 3kB -88.0 -87.98 -87.96 ... -82.44 -82.44 -82.44
    lat       (point) float64 3kB 29.96 29.96 29.96 29.96 ... 25.56 25.54 25.52
    x         (point) float64 3kB -1.759e+05 -1.739e+05 ... 3.554e+05 3.557e+05
    y         (point) float64 3kB 3.335e+06 3.335e+06 ... 2.826e+06 2.824e+06
    normal_x  (point) float64 3kB -0.0164 -0.01563 -0.01645 ... 0.9893 0.9907
    normal_y  (point) float64 3kB 0.9999 0.9999 0.9999 ... 0.1557 0.1459 0.1358

In [5]:
def attach_contour(ds, contour_ds):
    ds = ds.assign_coords(
        lon=("point", contour_ds["lon"].values),
        lat=("point", contour_ds["lat"].values),
        x=("point", contour_ds["x"].values),
        y=("point", contour_ds["y"].values),
    )
    ds["normal_x"] = ("point", contour_ds["normal_x"].values)
    ds["normal_y"] = ("point", contour_ds["normal_y"].values)
    return ds

### Read Hurrywave spectra

In [6]:
def read_hurrywave_2dspec(nc_path):
    ds0 = xr.open_dataset(nc_path, mask_and_scale=True, decode_cf=True)

    time = ds0["time"].values
    sigma = ds0["sigma"].values
    freq  = sigma / (2.0 * np.pi)          # Hz

    dir_from_deg = ds0["theta"].values     # degrees from

    # native shape: (time, stations, theta, sigma)
    S = ds0["point_spectrum2d"].values

    # follow your existing working conversion
    S = S * sigma[np.newaxis, np.newaxis, np.newaxis, :]

    # reorder to (time, point, freq, dir)
    S = np.transpose(S, (0, 1, 3, 2))

    ds = xr.Dataset(
        coords={
            "time": time,
            "point": np.arange(S.shape[1]),
            "freq": freq,
            "dir": dir_from_deg,
        },
        data_vars={
            "spec": (("time", "point", "freq", "dir"), S),
        },
        attrs={
            "model": "Hurrywave",
            "dir_convention": "from",
            "spec_units": "m^2/Hz/rad",
        }
    )

    ds = attach_contour(ds, contour_ds)

    return ds

In [7]:
data_dir = "F:/crs/proj/2025_NOPP_comparison/helene_deltares_wave_model_output/helene89pervmax/"
nc_path = data_dir+"hurrywave_sp2.nc"
ds_hw = read_hurrywave_2dspec(nc_path)
ds_hw

<xarray.Dataset> Size: 84MB
Dimensions:   (time: 115, point: 425, freq: 12, dir: 36)
Coordinates:
  * time      (time) datetime64[ns] 920B 2024-09-23 ... 2024-09-27T18:00:00
  * point     (point) int64 3kB 0 1 2 3 4 5 6 7 ... 418 419 420 421 422 423 424
  * freq      (freq) float32 48B 0.006366 0.008009 0.01008 ... 0.06325 0.07958
  * dir       (dir) float32 144B 5.0 15.0 25.0 35.0 ... 325.0 335.0 345.0 355.0
    lon       (point) float64 3kB -88.0 -87.98 -87.96 ... -82.44 -82.44 -82.44
    lat       (point) float64 3kB 29.96 29.96 29.96 29.96 ... 25.56 25.54 25.52
    x         (point) float64 3kB -1.759e+05 -1.739e+05 ... 3.554e+05 3.557e+05
    y         (point) float64 3kB 3.335e+06 3.335e+06 ... 2.826e+06 2.824e+06
Data variables:
    spec      (time, point, freq, dir) float32 84MB 0.0 0.0 ... 6.873e-05
    normal_x  (point) float64 3kB -0.0164 -0.01563 -0.01645 ... 0.9893 0.9907
    normal_y  (point) float64 3kB 0.9999 0.9999 0.9999 ... 0.1557 0.1459 0.1358
Attributes:
    model:           Hurrywave
    dir_convention:  from
    spec_units:      m^2/Hz/rad

### Read the ADCIRC SWAN .spc files

In [10]:
# kind of complicated to parse the ADCIRC files...need some subroutines.

_float_re = re.compile(r"[-+]?\d*\.\d+(?:[Ee][-+]?\d+)?|[-+]?\d+(?:[Ee][-+]?\d+)?")
_int_re   = re.compile(r"[-+]?\d+")
_time_re  = re.compile(r"^\d{8}\.\d{6}$")

def _floats_in_line(s):
    return [float(x) for x in _float_re.findall(s)]

def _ints_in_line(s):
    return [int(x) for x in _int_re.findall(s)]

def _parse_swan_time(tok):
    return datetime.strptime(tok, "%Y%m%d.%H%M%S")

def _read_number_block(lines, i0, n, kind="float"):
    vals = []
    i = i0
    getvals = _floats_in_line if kind == "float" else _ints_in_line
    while len(vals) < n:
        vals.extend(getvals(lines[i]))
        i += 1
    return np.array(vals[:n]), i

def read_swan_spec_vadens_factor(spc_path):
    """
    Read SWAN .spc file and return:
      lon, lat, freqs, dirs, times, E(nt,nf,nd) in m^2/Hz/rad
    """
    lines = Path(spc_path).read_text(errors="ignore").splitlines()

    # ---- header blocks ----
    for i, ln in enumerate(lines):
        s = ln.strip()

        if s.startswith("LONLAT"):
            lon, lat = _floats_in_line(lines[i+2])[:2]

        elif s.startswith("AFREQ"):
            nf = int(lines[i+1].split()[0])
            freqs, _ = _read_number_block(lines, i+2, nf, kind="float")
            freqs = freqs.astype(float)

        elif s.startswith("CDIR"):
            nd = int(lines[i+1].split()[0])
            dirs, _ = _read_number_block(lines, i+2, nd, kind="float")
            dirs = dirs.astype(float)

    # ---- time records ----
    times = []
    E_list = []

    i = 0
    while i < len(lines):
        tok = lines[i].strip().split()
        if tok and _time_re.match(tok[0]):
            t = _parse_swan_time(tok[0])

            while not lines[i].strip().startswith("FACTOR"):
                i += 1

            factor = _floats_in_line(lines[i+1])[0]
            A, i = _read_number_block(lines, i+2, nf*nd, kind="int")
            A = A.reshape(nf, nd)

            E = factor * A * (180.0 / np.pi)   # m^2/Hz/rad

            times.append(t)
            E_list.append(E)
        else:
            i += 1

    E = np.stack(E_list, axis=0)   # (nt, nf, nd)

    return {
        "lon": lon,
        "lat": lat,
        "freqs": freqs,
        "dirs": dirs,
        "times": times,
        "E": E,
    }

In [8]:
def read_adcirc_2dspec(spec_files):
    spec_files = list(spec_files)
    ds_list = []

    for ipt, fp in enumerate(spec_files):
        rec = read_swan_spec_vadens_factor(fp)

        time = np.asarray(rec["times"])
        freq = np.asarray(rec["freqs"])
        dir_from_deg = np.asarray(rec["dirs"])   # degrees from
        S = np.asarray(rec["E"])                 # (time, freq, dir)

        dsi = xr.Dataset(
            coords={
                "time": time,
                "point": [ipt],
                "freq": freq,
                "dir": dir_from_deg,
            },
            data_vars={
                "spec": (("time", "point", "freq", "dir"), S[:, np.newaxis, :, :]),
            }
        )
        ds_list.append(dsi)

    ds = xr.concat(ds_list, dim="point")
    ds.attrs["model"] = "ADCIRC/SWAN"
    ds.attrs["dir_convention"] = "from"
    ds.attrs["spec_units"] = "m^2/Hz/rad"

    ds = attach_contour(ds, contour_ds)

    return ds

In [11]:
spec_dir = "F:/crs/proj/2025_NOPP_comparison/helene_adcirc_model_results/spec_files/"
spec_files = sorted(glob.glob(spec_dir + "bnd*.spc"))   # bnd1.spc ... bnd425.spc
ds_ad = read_adcirc_2dspec(spec_files)
ds_ad

<xarray.Dataset> Size: 728MB
Dimensions:   (time: 145, point: 425, freq: 41, dir: 36)
Coordinates:
  * time      (time) datetime64[ns] 1kB 2024-09-25 ... 2024-09-28
  * point     (point) int64 3kB 0 1 2 3 4 5 6 7 ... 418 419 420 421 422 423 424
  * freq      (freq) float64 328B 0.0314 0.0345 0.038 ... 1.174 1.291 1.42
  * dir       (dir) float64 288B 5.0 15.0 25.0 35.0 ... 325.0 335.0 345.0 355.0
    lon       (point) float64 3kB -88.0 -87.98 -87.96 ... -82.44 -82.44 -82.44
    lat       (point) float64 3kB 29.96 29.96 29.96 29.96 ... 25.56 25.54 25.52
    x         (point) float64 3kB -1.759e+05 -1.739e+05 ... 3.554e+05 3.557e+05
    y         (point) float64 3kB 3.335e+06 3.335e+06 ... 2.826e+06 2.824e+06
Data variables:
    spec      (time, point, freq, dir) float64 728MB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    normal_x  (point) float64 3kB -0.0164 -0.01563 -0.01645 ... 0.9893 0.9907
    normal_y  (point) float64 3kB 0.9999 0.9999 0.9999 ... 0.1557 0.1459 0.1358
Attributes:
    model:           ADCIRC/SWAN
    dir_convention:  from
    spec_units:      m^2/Hz/rad

### Read the COAWST WW3 from THREDDS

In [14]:
def read_coawst_2dspec(url, engine="netcdf4"):
    ds0 = xr.open_dataset(url)

    time = ds0["time"].values
    freq = ds0["frequency"].values
    # note that directions are not sorted
    dir_from_deg = ds0["direction"].values   # degrees from

    # native shape: (time, station, frequency, direction)
    S = ds0["efth"].values

    ds = xr.Dataset(
        coords={
            "time": time,
            "point": np.arange(S.shape[1]),
            "freq": freq,
            "dir": dir_from_deg,
        },
        data_vars={
            "spec": (("time", "point", "freq", "dir"), S),
        },
        attrs={
            "model": "COAWST/WW3",
            "dir_convention": "from",
            "spec_units": "m^2/Hz/rad",
        }
    )
    
    ds = attach_contour(ds, contour_ds)

    return ds

In [15]:
dap_url = (
    "https://geoport.whoi.edu/thredds/dodsC/"
    "vortexfs1/usgs/Projects/Helene2024/helene77/Output_89pct/"
    "ww3.202409_spec.nc"
)
ds_cw = read_coawst_2dspec( dap_url )
ds_cw

<xarray.Dataset> Size: 237MB
Dimensions:   (time: 121, point: 425, freq: 32, dir: 36)
Coordinates:
  * time      (time) datetime64[ns] 968B 2024-09-24 ... 2024-09-29
  * point     (point) int64 3kB 0 1 2 3 4 5 6 7 ... 418 419 420 421 422 423 424
  * freq      (freq) float32 128B 0.038 0.0418 0.04598 ... 0.6028 0.6631 0.7294
  * dir       (dir) float32 144B 90.0 80.0 70.0 60.0 ... 130.0 120.0 110.0 100.0
    lon       (point) float64 3kB -88.0 -87.98 -87.96 ... -82.44 -82.44 -82.44
    lat       (point) float64 3kB 29.96 29.96 29.96 29.96 ... 25.56 25.54 25.52
    x         (point) float64 3kB -1.759e+05 -1.739e+05 ... 3.554e+05 3.557e+05
    y         (point) float64 3kB 3.335e+06 3.335e+06 ... 2.826e+06 2.824e+06
Data variables:
    spec      (time, point, freq, dir) float32 237MB 1.097e-21 ... 0.0001248
    normal_x  (point) float64 3kB -0.0164 -0.01563 -0.01645 ... 0.9893 0.9907
    normal_y  (point) float64 3kB 0.9999 0.9999 0.9999 ... 0.1557 0.1459 0.1358
Attributes:
    model:           COAWST/WW3
    dir_convention:  from
    spec_units:      m^2/Hz/rad

In [23]:
def calc_hm0(ds):
    f = ds["freq"].values
    th = np.deg2rad(ds["dir"].values)

    S = ds["spec"].values

    dth = np.gradient(th)
    Sf = np.sum(S * dth[np.newaxis, np.newaxis, np.newaxis, :], axis=-1)
    m0 = np.trapezoid(Sf, f, axis=-1)
    hm0 = 4.0 * np.sqrt(m0)

    ds["m0"] = (("time", "point"), m0)
    ds["hm0"] = (("time", "point"), hm0)
    ds["m0"].attrs["units"] = "m^2"
    ds["hm0"].attrs["units"] = "m"

    return ds

In [24]:
ds_hw = calc_hm0(ds_hw)
print( np.max( ds_hw['hm0'] ))

ds_ad = calc_hm0(ds_ad)
print( np.max( ds_ad['hm0'] ))

ds_cw = calc_hm0(ds_cw)
print( np.max( ds_cw['hm0'] ))

<xarray.DataArray 'hm0' ()> Size: 4B
array(0.7474227, dtype=float32)
<xarray.DataArray 'hm0' ()> Size: 8B
array(12.16494431)
<xarray.DataArray 'hm0' ()> Size: 4B
array(18.601803, dtype=float32)


C:\Users\csherwood\AppData\Local\Temp\1\ipykernel_36344\1858566815.py:10: RuntimeWarning: invalid value encountered in sqrt
  hm0 = 4.0 * np.sqrt(m0)
